# Habit / Energy / Focus Exploration

This notebook adds a small exploratory layer on top of the project database views.

It keeps the app semantics intact:

- `NULL` means unknown / not logged.
- Explicit zero values remain real logged zeroes.
- Completeness flags are used only to select interpretable subsets.

Timing matters here:

- sleep hours vs focus rating is a same-day morning relationship
- caffeine and exercise are usually logged for yesterday, so those analyses are lagged to the following morning

In [1]:
from pathlib import Path
import sys

import altair as alt
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.db import get_connection

alt.data_transformers.disable_max_rows()


def query_df(query: str) -> pd.DataFrame:
    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            rows = cursor.fetchall()
            columns = [column.name for column in cursor.description]
    return pd.DataFrame(rows, columns=columns)


def build_scatter_chart(
    df: pd.DataFrame,
    x_column: str,
    y_column: str,
    x_title: str,
    y_title: str,
    title: str,
):
    df = df.copy()
    if "checkin_date" in df.columns:
        df["checkin_date"] = pd.to_datetime(df["checkin_date"])
    df[x_column] = pd.to_numeric(df[x_column], errors="coerce")
    df[y_column] = pd.to_numeric(df[y_column], errors="coerce")

    return (
        alt.Chart(df)
        .mark_circle(size=90, color="#2E6F95")
        .encode(
            x=alt.X(f"{x_column}:Q", title=x_title),
            y=alt.Y(f"{y_column}:Q", title=y_title),
            tooltip=[
                alt.Tooltip("checkin_date:T", title="Date"),
                alt.Tooltip(f"{x_column}:Q", title=x_title),
                alt.Tooltip(f"{y_column}:Q", title=y_title),
            ],
        )
        .properties(title=title, height=320)
    )


In [2]:
analysis_df = query_df(
    """
    SELECT
        current_day.checkin_date,
        current_day.sleep_hours,
        current_day.sleep_quality,
        current_day.energy_rating,
        current_day.focus_rating,
        current_day.mood_rating,
        current_day.stress_rating,
        current_completeness.has_checkin,
        prior_day.deep_work_minutes AS prior_day_deep_work_minutes,
        prior_day.total_caffeine_mg AS prior_day_total_caffeine_mg,
        prior_day.total_exercise_minutes AS prior_day_total_exercise_minutes,
        prior_completeness.has_deep_work_entry AS prior_day_has_deep_work_entry,
        prior_completeness.has_caffeine_entry AS prior_day_has_caffeine_entry,
        prior_completeness.has_exercise_entry AS prior_day_has_exercise_entry
    FROM daily_metrics_vw AS current_day
    LEFT JOIN daily_completeness_vw AS current_completeness
        ON current_day.checkin_date = current_completeness.checkin_date
    LEFT JOIN daily_metrics_vw AS prior_day
        ON prior_day.checkin_date = current_day.checkin_date - 1
    LEFT JOIN daily_completeness_vw AS prior_completeness
        ON prior_completeness.checkin_date = current_day.checkin_date - 1
    ORDER BY current_day.checkin_date
    """
)

recent_completeness_df = query_df(
    """
    WITH recent_dates AS (
        SELECT
            generate_series(
                CURRENT_DATE - 13,
                CURRENT_DATE,
                INTERVAL '1 day'
            )::DATE AS checkin_date
    )
    SELECT
        recent_dates.checkin_date,
        COALESCE(dc.has_checkin, FALSE) AS has_checkin,
        COALESCE(dc.has_deep_work_entry, FALSE) AS has_deep_work_entry,
        COALESCE(dc.has_caffeine_entry, FALSE) AS has_caffeine_entry,
        COALESCE(dc.has_exercise_entry, FALSE) AS has_exercise_entry,
        COALESCE(dc.completed_sections, 0) AS completed_sections,
        COALESCE(dc.expected_sections, 4) AS expected_sections,
        COALESCE(dc.completeness_pct, 0.0) AS completeness_pct
    FROM recent_dates
    LEFT JOIN daily_completeness_vw AS dc
        ON recent_dates.checkin_date = dc.checkin_date
    ORDER BY recent_dates.checkin_date
    """
)

display(analysis_df.tail())


,checkin_date,sleep_hours,sleep_quality,energy_rating,focus_rating,mood_rating,stress_rating,has_checkin,prior_day_deep_work_minutes,prior_day_total_caffeine_mg,prior_day_total_exercise_minutes,prior_day_has_deep_work_entry,prior_day_has_caffeine_entry,prior_day_has_exercise_entry
2,2026-03-28,7.50,7.0,7.0,7.0,7.0,4.0,True,NaN,NaN,NaN,None,None,None
3,2026-03-29,6.75,7.0,7.0,7.0,7.0,4.0,True,60.0,205.0,45.0,True,True,True
4,2026-03-30,7.75,8.0,7.0,7.0,7.0,6.0,True,75.0,210.0,60.0,True,True,True
5,2026-04-01,None,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,None,None,None
6,2026-04-02,6.75,7.0,8.0,7.0,7.0,4.0,True,45.0,170.0,65.0,True,True,True


## Recent completeness overview

This mirrors the app's true 14-day calendar window, so dates with no row yet still show up as not logged.

In [3]:
recent_summary_df = pd.DataFrame(
    {
        "metric": [
            "Full data days",
            "Missing morning check-ins",
            "Missing deep work entries",
            "Missing caffeine entries",
            "Missing exercise entries",
        ],
        "value": [
            int((recent_completeness_df["completed_sections"] == recent_completeness_df["expected_sections"]).sum()),
            int((~recent_completeness_df["has_checkin"]).sum()),
            int((~recent_completeness_df["has_deep_work_entry"]).sum()),
            int((~recent_completeness_df["has_caffeine_entry"]).sum()),
            int((~recent_completeness_df["has_exercise_entry"]).sum()),
        ],
    }
)

display(recent_summary_df)
display(recent_completeness_df.sort_values("checkin_date", ascending=False))


,metric,value
0,Full data days,4
1,Missing morning check-ins,8
2,Missing deep work entries,9
3,Missing caffeine entries,9
4,Missing exercise entries,9


,checkin_date,has_checkin,has_deep_work_entry,has_caffeine_entry,has_exercise_entry,completed_sections,expected_sections,completeness_pct
13,2026-04-02,True,False,False,False,1,4,25.0
12,2026-04-01,False,True,True,True,3,4,75.0
11,2026-03-31,False,False,False,False,0,4,0.0
10,2026-03-30,True,False,False,False,1,4,25.0
9,2026-03-29,True,True,True,True,4,4,100.0
8,2026-03-28,True,True,True,True,4,4,100.0
7,2026-03-27,False,False,False,False,0,4,0.0
6,2026-03-26,True,True,True,True,4,4,100.0
5,2026-03-25,True,True,True,True,4,4,100.0
4,2026-03-24,False,False,False,False,0,4,0.0


The completeness flags are still a reporting layer, not a data rewrite. They help us keep unknown inputs out of the lagged comparisons while preserving explicit zero values as real observations.

## Relationship spot checks

These cells mirror the app's review analysis:

- same-day morning: sleep hours vs focus rating
- lagged: yesterday caffeine vs today sleep quality
- lagged: yesterday exercise vs today energy rating

In [4]:
sleep_focus_df = analysis_df[
    analysis_df["has_checkin"]
    & analysis_df["sleep_hours"].notna()
    & analysis_df["focus_rating"].notna()
][["checkin_date", "sleep_hours", "focus_rating"]].copy()

caffeine_sleep_df = analysis_df[
    analysis_df["prior_day_has_caffeine_entry"]
    & analysis_df["prior_day_total_caffeine_mg"].notna()
    & analysis_df["sleep_quality"].notna()
][["checkin_date", "prior_day_total_caffeine_mg", "sleep_quality"]].copy()

exercise_energy_df = analysis_df[
    analysis_df["prior_day_has_exercise_entry"]
    & analysis_df["prior_day_total_exercise_minutes"].notna()
    & analysis_df["energy_rating"].notna()
][["checkin_date", "prior_day_total_exercise_minutes", "energy_rating"]].copy()

relationship_summary_df = pd.DataFrame(
    {
        "analysis": [
            "Sleep hours vs same-day focus rating",
            "Yesterday total caffeine mg vs today sleep quality",
            "Yesterday total exercise minutes vs today energy rating",
        ],
        "usable_days": [
            len(sleep_focus_df),
            len(caffeine_sleep_df),
            len(exercise_energy_df),
        ],
        "corr": [
            sleep_focus_df["sleep_hours"].corr(sleep_focus_df["focus_rating"]),
            caffeine_sleep_df["prior_day_total_caffeine_mg"].corr(caffeine_sleep_df["sleep_quality"]),
            exercise_energy_df["prior_day_total_exercise_minutes"].corr(exercise_energy_df["energy_rating"]),
        ],
    }
).round(2)

display(relationship_summary_df)


,analysis,usable_days,corr
0,Sleep hours vs same-day focus rating,6,0.73
1,Yesterday total caffeine mg vs today sleep qua...,4,0.63
2,Yesterday total exercise minutes vs today ener...,4,0.90


In [5]:
build_scatter_chart(
    sleep_focus_df,
    x_column="sleep_hours",
    y_column="focus_rating",
    x_title="Sleep hours",
    y_title="Same-day focus rating",
    title="Sleep hours vs same-day focus rating",
)


alt.Chart(...)

This same-day slice uses only real morning check-ins, so placeholder parent rows created by child forms do not leak into the chart.

In [6]:
build_scatter_chart(
    caffeine_sleep_df,
    x_column="prior_day_total_caffeine_mg",
    y_column="sleep_quality",
    x_title="Yesterday total caffeine mg",
    y_title="Today sleep quality",
    title="Yesterday total caffeine mg vs today sleep quality",
)


alt.Chart(...)

This lagged slice uses yesterday's explicit caffeine logging status to decide whether the input is known. Explicit zero caffeine stays in the analysis, while unknown caffeine stays out.

In [7]:
build_scatter_chart(
    exercise_energy_df,
    x_column="prior_day_total_exercise_minutes",
    y_column="energy_rating",
    x_title="Yesterday total exercise minutes",
    y_title="Today energy rating",
    title="Yesterday total exercise minutes vs today energy rating",
)


alt.Chart(...)

This lagged exercise slice works the same way: yesterday's exercise must have been explicitly logged, but a logged zero still remains a valid prior-day observation.